### ACTION AND TRANSFORMATION

In [17]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("TransformationAndAction")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)

raw_data = [
    (1, "Laptop", "Electronics", 1200.0, 1, "Completed"),
    (2, "Desk Chair", "Furniture", 250.0, 2, "Completed"),
    (3, "Wireless Mouse", "Electronics", 35.0, 4, "Completed"),
    (4, "Monitor", "Electronics", None, 1, "Completed"),
    (5, "Standing Desk", "Furniture", 550.0, 1, "Completed"),
    (6, "Webcam", "Electronics", 85.0, -2, "Completed"),
    (7, "Notebook Pack", "Stationery", 12.0, 10, "Cancelled"),
    (8, "Desk Lamp", "Furniture", 45.0, 3, "Completed"),
    (9, "USB-C Hub", "Electronics", 60.0, 2, "Completed"),
    (10, "Pen Set", "Stationery", 15.0, 5, "Completed"),
]

bad_columns = [
    "Order ID",
    "Product Name",
    "Category",
    "Unit Price ($)",
    "Qty",
    "Order_Status",
]

# Step 1: Load data into a Spark DataFrame
df = spark.createDataFrame(raw_data, schema=bad_columns)

# Step 2: Rename messy columns ( Transformation)
df_renamed = (
    df.withColumnRenamed("Unit Price ($)", "unit_price")
    .withColumnRenamed("Qty", "quantity")
    .withColumnRenamed("Order_Status", "status")
)

# Step 3: Filter valid completed orders ( Transformation)
df_filtered = df_renamed.filter(
    (F.col("status") == "Completed")
    & (F.col("unit_price").isNotNull())
    & (F.col("quantity") > 0)
)

# Step 4: Calculate line revenue per row ( Transformation)
df_revenue = df_filtered.withColumn("revenue", F.col("unit_price") * F.col("quantity"))


#Step 5: Group by Category and aggregate ( Transformation )
df_grouped = df_revenue.groupBy("Category").agg(
    F.sum("revenue").alias("total_revenue")
)

# Step 6: Select final 2 columns ( Transformation )
final_df = df_grouped.select("Category", "total_revenue")

In [18]:
final_df.show() #Action

[Stage 0:>                                                          (0 + 2) / 2]

+-----------+-------------+
|   Category|total_revenue|
+-----------+-------------+
|Electronics|       1460.0|
|  Furniture|       1185.0|
| Stationery|         75.0|
+-----------+-------------+



In [20]:
spark.stop()